# 33 · 检索评估：Recall / MRR / NDCG

> 调 chunk、top-k、索引之前，先学会**量化检索好不好**。否则一切“感觉好”都是幻觉。

**本文件覆盖知识点**：Recall / Precision / Hit Rate / Recall@K / Precision@K / MRR / NDCG / MAP

先准备一份“测试集 + 人工标注的相关文档”，这是评估的前提。

In [ ]:
# 迷你评估集：query → 相关文档 id（人工标注）
test_set = [
    {'q':'什么是 RAG',        'rel':[0]},
    {'q':'怎么部署客服机器人', 'rel':[3]},
    {'q':'有哪些大模型',       'rel':[4]},
    {'q':'机器人怎么收费',     'rel':[5]},
]
corpus = ['RAG 检索增强生成', '向量数据库', '提示词工程', '支持公有云与私有化', 'qwen 系列', '分基础/专业/企业版']

In [ ]:
import numpy as np

def fake_retrieve(q, k=3):  # 桩检索器，生产换成真实检索
    qs = set(q)
    return list(np.argsort(-np.array([len(qs & set(d)) for d in corpus]))[:k])

# ---------- 指标实现 ----------
def recall_at_k(retrieved, rel, k):          # 召回的∩相关 / 总相关
    return len(set(retrieved[:k]) & set(rel)) / max(len(rel), 1)

def precision_at_k(retrieved, rel, k):       # 召回的∩相关 / k
    return len(set(retrieved[:k]) & set(rel)) / k

def hit_rate(retrieved, rel, k):             # Top-k 是否至少命中 1 条
    return 1.0 if set(retrieved[:k]) & set(rel) else 0.0

def mrr(retrieved, rel):                     # 第一个命中的倒数名次
    for r, d in enumerate(retrieved, 1):
        if d in rel: return 1.0 / r
    return 0.0

def ndcg(retrieved, rel, k):                 # 归一化折损累计增益
    dcg = sum(1/np.log2(i+1) for i, d in enumerate(retrieved[:k], 1) if d in rel)
    idcg = sum(1/np.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg / idcg if idcg else 0.0

# 汇总指标
k = 3
agg = {'recall':[], 'mrr':[], 'ndcg':[], 'hit':[]}
for t in test_set:
    r = fake_retrieve(t['q'], k)
    agg['recall'].append(recall_at_k(r, t['rel'], k))
    agg['mrr'].append(mrr(r, t['rel']))
    agg['ndcg'].append(ndcg(r, t['rel'], k))
    agg['hit'].append(hit_rate(r, t['rel'], k))

for name, v in agg.items():
    print(f'{name:8s} = {np.mean(v):.3f}')

## 指标怎么读

| 指标 | 侧重 | 一句话 |
|------|------|--------|
| **Recall@K** | 召回率 | 该找到的找到了多少（别漏） |
| **Precision@K** | 精确率 | 找到的有多少是对的（别错） |
| **Hit Rate** | 命中率 | 有没有至少找对一条 |
| **MRR** | 首位质量 | 第一个正确答案排多前 |
| **NDCG** | 排序质量 | 越相关排越前，加权计分 |
| **MAP** | 综合 | 多查询的平均精度均值 |

> **怎么用**：同一份测试集上换 chunk_size / top-k / 是否混合检索 / 是否重排，比较 Recall@K 与 NDCG，用数据决策而不是猜。

## 小结

- 先造**评估集**（人工标注相关文档），指标才有意义；
- Recall 管“别漏”，NDCG/MRR 管“排得好”，Precision 管“别错”；
- 检索指标是所有上游优化的“验收尺”。